# 12-00 TabPFN Intro und Setup

Dieses Notebook bildet den methodischen Einstieg in die finale TabPFN-Pipeline. Im Vordergrund steht noch kein Modelllauf, sondern die Prüfung, ob die für TabPFN verwendete Datenbasis sauber, reproduzierbar und konsistent mit dem chronologischen Split-Design ist.

Methodisch ist dieser Schritt wichtig, weil TabPFN später als pointwise Klassifikator auf drei binären Zielkanälen arbeitet. Wenn Features, Targets oder Split-Grenzen an dieser Stelle nicht stimmen, wären alle späteren Validierungs- und Testergebnisse schwer interpretierbar.

Kernregeln dieses Notebooks:

- Es werden drei binäre Targets vorbereitet: `Top5`, `Top10`, `Top20`.
- Es wird ausschließlich die aus `10-00_Model_Data_Prep.ipynb` exportierte Datenbasis aus `data/model_data/` genutzt.
- Die Validierungssaison 2023 und der Testzeitraum 2024/2025 bleiben klar getrennt.
- Dieses Notebook startet bewusst keine API-Läufe.

Die Ausgaben dieses Notebooks dienen damit als Kontrollbasis für alle nachfolgenden TabPFN-Notebooks.

## Leselogik dieses Notebooks

Dieses Notebook übernimmt die Rolle eines methodischen Prüfprotokolls für TabPFN: Es wird noch nichts trainiert, sondern dokumentiert, welche Datenbasis später in `12-02` und `12-03` überhaupt verwendet werden darf.

**Methodische Begründung:** TabPFN ist in dieser Arbeit kein isoliertes Experiment, sondern ein dritter Modellpfad neben EBM und XGBoost. Damit die Ergebnisse später vergleichbar sind, müssen Feature-Reihenfolge, Split-Grenzen, Target-Definitionen und Legacy-Artefakte vorab sichtbar fixiert werden. Ein Fehler an dieser Stelle würde sich in allen späteren API-Läufen fortsetzen.

**Interpretation der Outputs:** Die folgenden Tabellen sind deshalb keine dekorativen Checks. Sie sind die schriftliche Evidenz dafür, dass die neue TabPFN-Reihe mit `Top5`, `Top10` und `Top20` arbeitet, dass 2023 ausschließlich Validierung ist und dass 2024/2025 erst im finalen Test auftauchen.

## Methodische Langfassung: Warum `12-00` mehr als ein technisches Setup ist

Dieses Notebook ist als Datenvertrag für die gesamte TabPFN-Reihe zu lesen. Für TabPFN braucht es eine vergleichbare Vorarbeit wie bei XGBoost und EBM, nur liegt der Schwerpunkt anders: TabPFN bietet keine direkt interpretierbaren Feature-Funktionen, sondern arbeitet als vortrainierter Klassifikator auf der vorbereiteten Datenmatrix. Deshalb muss die Nachvollziehbarkeit stärker über Daten, Splits, Targets, Thresholds und Output-Konventionen entstehen.

**Datenvertrag:** Die 17 Features aus `data/model_data/X_*.pkl` sind die einzige Eingabematrix für die neuen TabPFN-Notebooks. Es wird keine zusätzliche Feature-Selektion im TabPFN-Pfad durchgeführt, weil sonst die Vergleichbarkeit zu EBM und XGBoost unklar würde. Die Feature-Reihenfolge ist dabei nicht nur eine technische Kleinigkeit: Ein tabellarisches Modell verarbeitet Spalten positionell und semantisch. Wenn dieselben Spalten in anderer Reihenfolge ankommen, wäre der Lauf zwar syntaktisch möglich, aber fachlich ungültig.

**Target-Vertrag:** Die neue Methode arbeitet nicht mehr mit einem ordinalen Modellziel. Stattdessen werden drei binäre Zielkanäle genutzt: `Top5`, `Top10` und `Top20`. Diese Kanäle sind kumulativ. Ein Top5-Fahrer ist automatisch auch Top10 und Top20. Daraus entstehen drei Wahrscheinlichkeiten pro Fahrer, die wie im EBM-Pfad zu einem Frank-&-Hall-Score addiert werden.

**Split-Vertrag:** Die zeitliche Trennung ist die wichtigste Schutzmaßnahme gegen Data Leakage. `train_selection` umfasst Daten bis einschließlich 2022. `validation` ist ausschließlich 2023 und dient der Auswahl der normalen Classifier-Parameter sowie der Kalibrierung des Frank-&-Hall-Top10-Thresholds. `test_original` umfasst 2024 und 2025 und bleibt bis zum finalen Notebook unberührt. Das finale Training darf nach abgeschlossener Hyperparameter- und Threshold-Wahl `train + validation` verwenden, weil 2023 dann nicht mehr als Entscheidungsquelle genutzt wird, sondern Teil des bekannten historischen Kontexts vor dem Zukunftstest ist.

**Output-Vertrag:** Neue Artefakte werden unter `data/processed/tabpfn/` geschrieben. Kompakte, modellvergleichbare Ergebnisse landen zusätzlich unter `data/models/`. Alte TabPFN-Artefakte bleiben sichtbar, werden aber nicht als neue Entscheidungsgrundlage verwendet. Diese Trennung ist besonders wichtig, weil ältere Notebooks teilweise andere Zielvariablen oder Modelllogiken ausprobiert haben.

**Wie die Tabellen zu lesen sind:** Die Tabellen in diesem Notebook sind bewusst nüchtern gehalten: Pfade, Features, Splits, Target-Verteilungen und Legacy-Rollen. Gerade diese Nüchternheit macht sie wertvoll. Sie belegen, dass spätere API- oder Cache-Läufe nicht auf einer impliziten Annahme beruhen, sondern auf einem dokumentierten Datenstand.


## Einordnung von TabPFN

TabPFN wird hier als vortrainiertes Foundation Model für tabellarische Daten eingesetzt. Im Unterschied zu den vorherigen Modellfamilien wird kein klassischer Baum-Ensemble-Prozess und kein explizit additives Erklärmodell trainiert, sondern ein `TabPFNClassifier`, der Trainingsbeispiele als Kontext nutzt und daraus Wahrscheinlichkeiten für neue Fahrer-Etappen-Kombinationen ableitet.

**Methodische Begründung:** Für das Radsportprojekt ist TabPFN interessant, weil die Features heterogen sind: Etappenprofil, Fahrerphysiologie, Wetter, Teamstärke und vergangene Leistungsindikatoren stehen gemeinsam in einer kompaktem tabellarischen Datenmatrix. TabPFN kann solche Tabellen direkt verarbeiten und liefert Wahrscheinlichkeiten, die später rankingfähig gemacht werden.

**Abgrenzung zu EBM und XGBoost:** Das EBM-Notebook bleibt die interpretierbare Glass-Box-Basis mit additiven Effekten und ausgewählten Interaktionen. XGBoost ist die starke klassische ML-Baseline für nichtlineare Muster. TabPFN ergänzt diese Perspektive als vortrainierter, probabilistischer Klassifikator. Dafür ist TabPFN weniger direkt erklärbar und muss deshalb besonders sauber über Splits, Targets, Metriken und Cache-Artefakte dokumentiert werden.

**Rankinglogik:** Das eigentliche Projektziel ist ein Etappenranking. TabPFN wird hier aber nicht direkt als Rankingmodell verwendet. Stattdessen werden drei binäre Klassifikatoren trainiert: `Top5`, `Top10` und `Top20`. Diese drei Targets bilden sportlich sinnvolle Schwellen ab und erzeugen Wahrscheinlichkeiten, die später zu genau einem finalen Ranking-Score kombiniert werden.

**Limitation:** Die drei binären Modelle werden separat trainiert. Dadurch können die kumulativen Wahrscheinlichkeiten im Einzelfall widersprüchlich sein, etwa wenn `P(Top5)` größer als `P(Top10)` ist. Diese Fälle werden im finalen Notebook sichtbar gemacht, aber der verbindliche Ranking-Score bleibt die rohe Summe der drei Top-K-Wahrscheinlichkeiten.

### Warum so viel Setup-Dokumentation nötig ist

Im Gegensatz zum EBM liefert TabPFN keine interpretierbaren Shape-Funktionen, Feature-Plots oder direkte Feature-Importance, die später als erklärender Anker dienen könnten. Die Nachvollziehbarkeit entsteht deshalb stärker über das experimentelle Design: Welche Daten darf das Modell sehen? Welche Zielvariable wird optimiert? Welche Metrik entscheidet? Welche Artefakte sind nur historischer Kontext?

Für das Radsport-Ranking ist diese Trennung besonders wichtig, weil die einzelnen Fahrerbeobachtungen nicht unabhängig im sportlichen Sinn sind. Innerhalb einer Etappe konkurrieren alle Fahrer um dieselben Plätze. Trotzdem wird TabPFN als binärer Klassifikator trainiert. Das Notebook macht diese Modellvereinfachung früh sichtbar, damit spätere Rankingmetriken nicht fälschlicherweise als direkt trainierte Rankingziele gelesen werden.

## Alter Plan und neuer Plan

Die neue Notebook-Reihe ersetzt ältere TabPFN-Versuche nicht historisch, trennt sie aber klar von der finalen Modelllogik. Alte Artefakte dürfen als Kontext gelesen werden, fließen aber nicht mehr in die neue Hyperparameterwahl ein.

| Bereich | Alter Plan | Neuer Plan |
| --- | --- | --- |
| Zielvariable | `target_ordinal` | `y_top5`, `y_top10`, `y_top20` |
| Modellfamilien | mehrere Modellpfade | nur `TabPFNClassifier` |
| Validierung | Rankingmetrik wie `ndcg_at_10` | `roc_auc` auf `Top10` plus Top10-Threshold auf 2023 |
| Thinking | teilweise im Tuning geplant | nur finaler Testvergleich mit Fingerprint-Sidecars |
| Finale Modelle | ein ausgewähltes Modell | drei binäre Classifier mit gleichen Parametern |
| Finale Klassifikation | nicht klar getrennt | native Einzelkanäle, natives Ensemble und validierter Ensemble-Threshold |
| Finale Rankings | einzelne Scores möglich | nur `score_topk_raw_sum` |
| Finale Evaluation | Rankingmetriken | Klassifikationsmetriken, Konfusionsmatrizen, Rankingmetriken, MAP und TDF-Case-Study |

**Schutz vor Leakage:** Die Modellauswahl und Threshold-Kalibrierung finden ausschließlich auf dem 2023er Validierungssplit statt. Der Testzeitraum 2024/2025 bleibt bis zum finalen Notebook unberührt.

**Interpretation:** Diese Tabelle ist die methodische Leitplanke für die gesamte Reihe. Wenn spätere Zellen alte `target_ordinal`-Dateien oder alternative Ranking-Scores erwähnen, dann nur zur historischen Einordnung, nicht als neue Entscheidungsgrundlage.

### Konsequenz für die spätere Interpretation

Die alte ordinale Zielvariable bleibt als historische Entwicklungsstufe erklärbar, aber nicht entscheidungsrelevant. Der neue Plan ist enger und dadurch sauberer: `Top10` wählt die normalen Classifier-Parameter, anschließend werden dieselben Parameter auf `Top5`, `Top10` und `Top20` übertragen. In `12-02` wird daraus zusätzlich ein validierter Frank-&-Hall-Threshold für `actual_top10` bestimmt. Erst danach erzeugt `12-03` den finalen Zukunftstest.

Diese Designentscheidung ist bewusst konservativ. Sie vermeidet, dass verschiedene Targets jeweils eigene, schwer vergleichbare Hyperparameter erhalten. Gleichzeitig bleibt eine Limitation: Ein gemeinsamer Parametersatz kann für `Top5` oder `Top20` suboptimal sein. Diese mögliche Schwäche wird akzeptiert, weil die Modellwahl nicht auf dem Zukunftstest 2024/2025 stattfinden darf.


## Chronologisches Split-Design, Threshold und kombinierter Score

Alle folgenden Notebooks verwenden dieselbe zeitliche Trennung: Training bis einschließlich 2022, Validierung auf 2023 und finaler Test auf 2024/2025. Dadurch wird vermieden, dass Informationen aus zukünftigen Saisons in die Modellwahl gelangen.

`12-02` hat zwei Validierungsaufgaben. Erstens wird die normale TabPFN-Konfiguration auf `Top10` ausgewählt. Zweitens werden mit dieser Konfiguration `Top5`, `Top10` und `Top20` auf 2023 vorhergesagt, zu einem Frank-&-Hall-Score addiert und ein F1-optimaler Threshold für `actual_top10` kalibriert. Dieser Threshold wird in `tabpfn_selected_classifier_params.json` gespeichert.

Das finale Ranking entsteht erst in `12-03`. Dort werden drei Rohwahrscheinlichkeiten pro Fahrer addiert:

```python
score_topk_raw_sum = p_top5_raw + p_top10_raw + p_top20_raw
```

Aus diesem Score wird je `stage_id` genau ein Ranking gebildet. Einzelrankings nach `p_top5_raw`, `p_top10_raw` oder `p_top20_raw` werden bewusst nicht als finale Modelloutputs erzeugt.

**Limitation:** Der Score ist kein kausal interpretierbarer Leistungswert. Er ist eine pragmatische, probabilistische Aggregation der drei Top-K-Signale und muss deshalb über Rankingmetriken wie `NDCG@5/10/20`, Winner-Hit-Rates und `MAP` bewertet werden. Die binäre Top10-Klassifikation wird separat über Konfusionsmatrizen und den validierten Threshold bewertet.

### Was später nicht passieren darf

Aus den drei Wahrscheinlichkeiten dürfen keine konkurrierenden finalen Rankings gebaut werden. Ebenso darf in `12-03` kein neuer Threshold auf 2024/2025 gesucht werden. Einzelne Spalten wie `p_top5_raw` oder `p_top20_raw` sind diagnostisch interessant, aber sie definieren nicht das finale Ranking.


In [ ]:
# Dieser Codeblock richtet die Arbeitsumgebung ein und sucht die Projektwurzel sichtbar und reproduzierbar.

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Die Projektwurzel wird ohne Hilfsfunktion gesucht, damit die Pfadlogik direkt im Notebook lesbar bleibt.
start_path = Path.cwd()
PROJECT_ROOT = None
for candidate in [start_path, *start_path.parents]:
    has_model_data = (candidate / "data" / "model_data").exists()
    has_notebooks = (candidate / "src" / "Notebooks").exists()
    if has_model_data and has_notebooks:
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Projektwurzel mit data/model_data und src/Notebooks nicht gefunden.")

MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABPFN_DIR = PROCESSED_DIR / "tabpfn"
LEGACY_DIR = PROCESSED_DIR / "tabpfn_3_experiments"

path_overview = pd.DataFrame([
    {"name": "PROJECT_ROOT", "path": str(PROJECT_ROOT), "exists": PROJECT_ROOT.exists()},
    {"name": "MODEL_DATA_DIR", "path": str(MODEL_DATA_DIR), "exists": MODEL_DATA_DIR.exists()},
    {"name": "TABPFN_DIR", "path": str(TABPFN_DIR), "exists": TABPFN_DIR.exists()},
    {"name": "LEGACY_DIR", "path": str(LEGACY_DIR), "exists": LEGACY_DIR.exists()},
])

print("=" * 88)
print("TABPFN SETUP: Pfadprüfung")
print("=" * 88)
display(path_overview)

### Interpretation der Pfadprüfung

Die Pfadtabelle zeigt, ob das Notebook im erwarteten Projektkontext läuft. Besonders wichtig ist `MODEL_DATA_DIR`, weil alle späteren Notebook-Schritte aus genau diesem Ordner lesen. Wenn hier ein `False` auftaucht, ist nicht die Modellmethode das Problem, sondern die Ausführungsumgebung.

Für TabPFN ist dieser Schritt wichtig, weil spätere API-Läufe teuer oder zeitaufwendig sein können. Vor solchen Läufen muss klar sein, dass die lokale Datenbasis stimmt.

In [ ]:
# Die verbindliche Feature- und Target-Konfiguration wird explizit im Notebook festgehalten.

FEATURE_COLUMNS = [
    "distance",
    "vertical_meters",
    "stage_nr",
    "team_tier",
    "age_at_race",
    "rider_bmi",
    "wind_stability_index",
    "weather_temp_mean",
    "weather_temp_trend",
    "weather_rain_prob_mean",
    "weather_precipitation_mean",
    "weather_humidity_mean",
    "gradient_final_km",
    "lag_rider_points_season",
    "lag_rider_rank_season",
    "lag_race_competitiveness_median",
    "lag_team_power_index",
]

TARGET_CONFIGS = {
    "top5": {
        "label": "Top5",
        "train_file": "y_top5_train.pkl",
        "valid_file": "y_top5_valid.pkl",
        "test_file": "y_top5_test.pkl",
        "score_col": "p_top5_raw",
        "actual_col": "actual_top5",
    },
    "top10": {
        "label": "Top10",
        "train_file": "y_top10_train.pkl",
        "valid_file": "y_top10_valid.pkl",
        "test_file": "y_top10_test.pkl",
        "score_col": "p_top10_raw",
        "actual_col": "actual_top10",
    },
    "top20": {
        "label": "Top20",
        "train_file": "y_top20_train.pkl",
        "valid_file": "y_top20_valid.pkl",
        "test_file": "y_top20_test.pkl",
        "score_col": "p_top20_raw",
        "actual_col": "actual_top20",
    },
}

EXPECTED_SHAPES = {
    "train": (169349, 17),
    "valid": (8897, 17),
    "test": (17802, 17),
}

feature_columns = pd.DataFrame({
    "position": range(1, len(FEATURE_COLUMNS) + 1),
    "feature": FEATURE_COLUMNS,
})

target_config_table = pd.DataFrame([
    {
        "target": target,
        "label": cfg["label"],
        "train_file": cfg["train_file"],
        "valid_file": cfg["valid_file"],
        "test_file": cfg["test_file"],
        "raw_probability_column": cfg["score_col"],
        "actual_column": cfg["actual_col"],
    }
    for target, cfg in TARGET_CONFIGS.items()
])

print("=" * 88)
print("TABPFN SETUP: Features und Targets")
print("=" * 88)
print(f"Anzahl Features: {len(FEATURE_COLUMNS)}")
display(feature_columns)
display(target_config_table)

### Interpretation der Feature- und Target-Konfiguration

Die Feature-Tabelle ist die verbindliche Spaltenreihenfolge für alle TabPFN-Läufe. Tabellarische Modelle reagieren empfindlich darauf, wenn Trainings- und Testdaten zwar dieselben Spalten, aber eine andere Reihenfolge besitzen. Die Target-Tabelle macht zusätzlich sichtbar, dass nicht mehr mit einem ordinalen Modellziel gearbeitet wird, sondern mit drei separaten binären Zielkanälen.

## Daten laden

In diesem Abschnitt werden alle Features, Targets, Gruppeninformationen und Metadaten aus `data/model_data/` geladen. Die Pfade werden über die Projektwurzel bestimmt, damit das Notebook unabhängig davon funktioniert, aus welchem Arbeitsverzeichnis es geöffnet wird.

Die geladenen Objekte bilden die gemeinsame Grundlage für Tuning, finale Evaluation und Case Studies. Deshalb wird hier bewusst breit geladen, auch wenn einzelne Objekte erst in späteren Notebooks modellrelevant werden.

Nach der Codezelle sollte geprüft werden, ob alle erwarteten Pickle-Dateien gefunden wurden und ob die Objektgrößen plausibel wirken.

In [ ]:
# Leseschlüssel für die Datenladezelle:
# 1. Zuerst werden Feature-Matrizen, Gruppen und Metadaten geladen.
# 2. Danach werden alle drei binären Targets in einem gemeinsamen Dictionary abgelegt.
# 3. Die abschließende Tabelle erklärt, welche Objektgruppe später welche methodische Rolle hat.

# In diesem Block werden die vorbereiteten Pickle-Dateien direkt geladen und anschließend als Kontrolltabelle dokumentiert.

SETUP_DIR = TABPFN_DIR / "00_setup"
SETUP_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_pickle(MODEL_DATA_DIR / "X_train.pkl")
X_valid = pd.read_pickle(MODEL_DATA_DIR / "X_valid.pkl")
X_test = pd.read_pickle(MODEL_DATA_DIR / "X_test.pkl")

groups_train = pd.read_pickle(MODEL_DATA_DIR / "groups_train.pkl")
groups_valid = pd.read_pickle(MODEL_DATA_DIR / "groups_valid.pkl")
groups_test = pd.read_pickle(MODEL_DATA_DIR / "groups_test.pkl")

meta_valid = pd.read_pickle(MODEL_DATA_DIR / "meta_valid.pkl")
meta_test = pd.read_pickle(MODEL_DATA_DIR / "meta_test.pkl")

y_rank_train = pd.read_pickle(MODEL_DATA_DIR / "y_rank_train.pkl")
y_rank_valid = pd.read_pickle(MODEL_DATA_DIR / "y_rank_valid.pkl")
y_rank_test = pd.read_pickle(MODEL_DATA_DIR / "y_rank_test.pkl")

y_targets = {}
for target, cfg in TARGET_CONFIGS.items():
    y_targets[(target, "train")] = pd.read_pickle(MODEL_DATA_DIR / cfg["train_file"])
    y_targets[(target, "valid")] = pd.read_pickle(MODEL_DATA_DIR / cfg["valid_file"])
    y_targets[(target, "test")] = pd.read_pickle(MODEL_DATA_DIR / cfg["test_file"])

loaded_objects = pd.DataFrame([
    {"object": "X_train", "rows": len(X_train), "columns": X_train.shape[1], "role": "Training bis 2022"},
    {"object": "X_valid", "rows": len(X_valid), "columns": X_valid.shape[1], "role": "Validierung 2023"},
    {"object": "X_test", "rows": len(X_test), "columns": X_test.shape[1], "role": "Finaler Test 2024/2025"},
    {"object": "groups_train", "rows": len(groups_train), "columns": 1, "role": "Stage-Gruppen Training"},
    {"object": "groups_valid", "rows": len(groups_valid), "columns": 1, "role": "Stage-Gruppen Validierung"},
    {"object": "groups_test", "rows": len(groups_test), "columns": 1, "role": "Stage-Gruppen Test"},
    {"object": "meta_valid", "rows": len(meta_valid), "columns": meta_valid.shape[1], "role": "Metadaten Validierung"},
    {"object": "meta_test", "rows": len(meta_test), "columns": meta_test.shape[1], "role": "Metadaten Test"},
])

print("=" * 88)
print("TABPFN SETUP: Geladene Modelldaten")
print("=" * 88)
display(loaded_objects)

### Interpretation der geladenen Objekte

Die Kontrolltabelle trennt Modellfeatures, Gruppeninformationen, Metadaten und Ränge. Diese Trennung hilft beim Lesen der späteren Notebooks: Features gehen in TabPFN ein, Gruppen definieren die etappenweise Rankingauswertung, Metadaten machen die Case Study lesbar und `y_rank_*` liefert die sportliche Wahrheit für Rankingmetriken.

## Feature- und Split-Checks

Dieser Abschnitt prüft, ob die exportierten Feature-Spalten und Split-Größen dem geplanten Modellsetup entsprechen. Besonders wichtig ist die feste Feature-Reihenfolge, weil TabPFN die tabellarischen Eingaben genau in dieser Struktur erhält.

Abweichende Zeilenzahlen werden nicht sofort als Fehler behandelt, sondern als Warnung ausgegeben. Das ist sinnvoll, falls `10-00_Model_Data_Prep.ipynb` später neu exportiert wurde. Inhaltlich muss eine solche Warnung aber immer geprüft werden, bevor neue Modellläufe gestartet werden.

Die zentrale Interpretationsfrage nach dieser Zelle lautet: Arbeiten die folgenden Notebooks auf derselben Datenbasis, die in der Methode beschrieben wird?

In [ ]:
# Diese Checks stellen sicher, dass Feature-Reihenfolge, Split-Grenzen und Metadaten zur dokumentierten Methode passen.

assert len(FEATURE_COLUMNS) == 17
assert list(X_train.columns) == FEATURE_COLUMNS
assert list(X_valid.columns) == FEATURE_COLUMNS
assert list(X_test.columns) == FEATURE_COLUMNS

check_rows = []
for split_name, X in [("train", X_train), ("valid", X_valid), ("test", X_test)]:
    expected_shape = EXPECTED_SHAPES[split_name]
    shape_ok = X.shape == expected_shape
    if not shape_ok:
        warnings.warn(f"{split_name}: erwartete Shape {expected_shape}, gefunden {X.shape}")
    check_rows.append({
        "check": f"{split_name}_shape",
        "expected": str(expected_shape),
        "observed": str(X.shape),
        "status": "ok" if shape_ok else "warning",
    })

valid_years = set(meta_valid["meta_year"].unique())
test_years = set(meta_test["meta_year"].unique())
assert valid_years == {2023}
assert test_years.issubset({2024, 2025})
assert len(groups_train) == len(X_train)
assert len(groups_valid) == len(X_valid)
assert len(groups_test) == len(X_test)

check_rows.extend([
    {"check": "validation_year", "expected": "{2023}", "observed": str(valid_years), "status": "ok"},
    {"check": "test_years", "expected": "subset {2024, 2025}", "observed": str(test_years), "status": "ok"},
    {"check": "groups_train_length", "expected": len(X_train), "observed": len(groups_train), "status": "ok"},
    {"check": "groups_valid_length", "expected": len(X_valid), "observed": len(groups_valid), "status": "ok"},
    {"check": "groups_test_length", "expected": len(X_test), "observed": len(groups_test), "status": "ok"},
])

feature_columns.to_csv(SETUP_DIR / "feature_columns.csv", index=False)
check_table = pd.DataFrame(check_rows)

print("=" * 88)
print("TABPFN SETUP: Split- und Feature-Checks")
print("=" * 88)
display(check_table)
display(feature_columns)

### Interpretation der Checks

Ein `ok` in dieser Tabelle bedeutet nicht, dass das Modell gut ist, sondern dass die methodische Grundlage belastbar ist. Shape-Warnungen wären nicht automatisch fatal, müssten aber in der Thesis erklärt werden, weil sie auf einen neu exportierten Modelldatenstand hinweisen könnten.

## Split-Übersicht schreiben

Die Split-Übersicht fasst die Anzahl der Zeilen, Features, Stages und Jahre pro Datensplit zusammen. Damit wird die zeitliche Trennung transparent dokumentiert.

Diese Tabelle ist vor allem für die spätere Thesis wichtig: Sie zeigt auf einen Blick, dass Training, Validierung und Test nicht zufällig gemischt wurden, sondern chronologisch getrennt bleiben.

Nach der Codezelle sollte kontrolliert werden, ob `validation` nur 2023 enthält und ob `test` ausschließlich 2024/2025 umfasst.

In [ ]:
# Die Split-Übersicht wird als kompakte Evidenztabelle exportiert.

train_final_groups = pd.concat([pd.Series(groups_train), pd.Series(groups_valid)], axis=0)
split_overview = pd.DataFrame([
    {
        "split": "train_selection",
        "rows": len(X_train),
        "features": X_train.shape[1],
        "stages": pd.Series(groups_train).nunique(),
        "years": "<=2022",
        "method_role": "Training für Top10-Validierung",
    },
    {
        "split": "validation",
        "rows": len(X_valid),
        "features": X_valid.shape[1],
        "stages": pd.Series(groups_valid).nunique(),
        "years": ", ".join(map(str, sorted(meta_valid["meta_year"].dropna().unique()))),
        "method_role": "Hyperparameterwahl nur hier",
    },
    {
        "split": "test_original",
        "rows": len(X_test),
        "features": X_test.shape[1],
        "stages": pd.Series(groups_test).nunique(),
        "years": ", ".join(map(str, sorted(meta_test["meta_year"].dropna().unique()))),
        "method_role": "Finaler Zukunftstest",
    },
    {
        "split": "train_final",
        "rows": len(pd.concat([X_train, X_valid], axis=0)),
        "features": X_train.shape[1],
        "stages": train_final_groups.nunique(),
        "years": "<=2023",
        "method_role": "Finales Training vor Test",
    },
])

split_overview.to_csv(SETUP_DIR / "model_data_split_overview.csv", index=False)

print("=" * 88)
print("TABPFN SETUP: Chronologische Split-Übersicht")
print("=" * 88)
display(split_overview)

### Interpretation der Split-Übersicht

Diese Tabelle ist der zentrale Leakage-Schutz des TabPFN-Setups. Die Hyperparameterwahl darf nur über `validation` laufen. Der Split `test_original` wird erst in `12-03` genutzt und simuliert den echten Zukunftstest auf noch nicht zur Auswahl verwendeten Saisons.

## Target-Verteilungen schreiben

Hier werden die positiven Fälle je Target und Split gezählt. Das ist methodisch relevant, weil `Top5`, `Top10` und `Top20` stark unbalancierte binäre Klassifikationsprobleme sind.

Die positive Rate hilft später bei der Interpretation von `roc_auc` und `average_precision`: Eine gute ROC-AUC kann bei stark unbalancierten Daten anders wirken als eine gute Precision-Recall-Leistung.

Die Tabelle dokumentiert deshalb nicht nur technische Größen, sondern auch die Grundschwierigkeit des sportlichen Prognoseproblems.

In [ ]:
# Leseschlüssel für die Target-Verteilung:
# 1. Jede Kombination aus Split und Target wird separat gezählt.
# 2. Positive Fälle zeigen die Seltenheit der jeweiligen Top-K-Zone.
# 3. Die Prozentwerte helfen später bei der Interpretation von ROC-AUC und Average Precision.

# Die Target-Verteilungen quantifizieren das Klassenungleichgewicht in Top5, Top10 und Top20.

rows = []
for target in TARGET_CONFIGS:
    for split, y, groups, meta, years_label in [
        ("train", y_targets[(target, "train")], groups_train, None, "<=2022"),
        ("validation", y_targets[(target, "valid")], groups_valid, meta_valid, None),
        ("test", y_targets[(target, "test")], groups_test, meta_test, None),
    ]:
        y_series = pd.Series(y)
        positives = int((y_series == 1).sum())
        negatives = int((y_series == 0).sum())
        if meta is not None and "meta_year" in meta.columns:
            years = ", ".join(map(str, sorted(meta["meta_year"].dropna().unique())))
        else:
            years = years_label
        rows.append({
            "split": split,
            "target": target,
            "rows": len(y_series),
            "positives": positives,
            "negatives": negatives,
            "positive_rate": positives / len(y_series) if len(y_series) else np.nan,
            "positive_rate_percent": 100 * positives / len(y_series) if len(y_series) else np.nan,
            "stages": pd.Series(groups).nunique(),
            "years": years,
        })

target_distribution = pd.DataFrame(rows)
target_distribution.to_csv(SETUP_DIR / "target_distribution_by_split.csv", index=False)

target_pivot = target_distribution.pivot(index="target", columns="split", values="positive_rate_percent").reset_index()

print("=" * 88)
print("TABPFN SETUP: Klassenungleichgewicht der drei binären Targets")
print("=" * 88)
display(target_distribution)
print("Positive Rate in Prozent, kompakt nach Target:")
display(target_pivot)

### Interpretation der Target-Verteilungen

Die positiven Raten zeigen, warum einfache Accuracy-Werte hier ungeeignet wären. `Top5` ist besonders selten, `Top20` etwas weniger extrem, aber alle drei Targets sind deutlich unbalanciert. Deshalb werden in `12-02` neben `roc_auc` auch `average_precision` und später Rankingmetriken betrachtet.

## Legacy-Artefakte einordnen

Dieser Abschnitt ordnet alte TabPFN-Dateien und frühere Ergebnisordner historisch ein. Die alten Artefakte dürfen als Kontext gelesen werden, fließen aber nicht mehr direkt in die neue Modellwahl ein.

Wichtig ist diese Trennung, weil der neue Ansatz ausschließlich mit `TabPFNClassifier`, drei binären Targets und der Top10-Validierung über `roc_auc` arbeitet. Legacy-Ergebnisse mit anderen Zielvariablen oder Scores sind deshalb nur Vergleichsmaterial.

Nach der Codezelle sollte klar sein, welche Dateien noch als Referenz existieren und welche Rolle sie im neuen Plan spielen.

In [ ]:
# Die Legacy-Tabelle trennt historische Referenzen von der neuen, verbindlichen TabPFN-Modelllogik.

legacy_rows = [
    {"artifact": "src/Notebooks/tabpfn_alt/12-00-01_Versuch_1_TabPFN_3.ipynb", "role": "historische lokale Top10-Tests"},
    {"artifact": "src/Notebooks/tabpfn_alt/12-00-02_tabpfn-client_Versuch.ipynb", "role": "historischer API-Versuch"},
    {"artifact": "src/Notebooks/tabpfn_alt/12-02-02_TabPFN_Client_Learning_Curve_Validation_2023.ipynb", "role": "Legacy-Learning-Curve, nur Kontext"},
    {"artifact": "src/Notebooks/tabpfn_alt/12-03-01_TabPFN_Classifier.ipynb", "role": "Referenz für binären Classifier-Code"},
    {"artifact": "data/processed/tabpfn_3_experiments/", "role": "Legacy-Cache, nicht neue Ergebnisquelle"},
    {"artifact": "data/processed/tabpfn_top10_metrics.csv", "role": "historische Top10-Metrikdatei"},
]

legacy_artifacts = pd.DataFrame(legacy_rows)
exists_values = []
for artifact in legacy_artifacts["artifact"]:
    exists_values.append((PROJECT_ROOT / artifact).exists())
legacy_artifacts["exists"] = exists_values
legacy_artifacts["new_plan_use"] = np.where(legacy_artifacts["exists"], "nur lesen/kontextualisieren", "nicht verfügbar")
legacy_artifacts.to_csv(SETUP_DIR / "legacy_artifact_overview.csv", index=False)

print("=" * 88)
print("TABPFN SETUP: Legacy-Artefakte und neue Rolle")
print("=" * 88)
display(legacy_artifacts)

### Interpretation der Legacy-Tabelle

Legacy-Artefakte sind nicht wertlos, aber sie haben eine andere Rolle als neue Modelloutputs. Sie dürfen helfen, frühere Entscheidungen zu verstehen oder vorhandene Caches einzuordnen. Sie dürfen jedoch nicht die neue Top10-Validierung ersetzen und nicht wieder `target_ordinal` als Modellziel einschleusen.

## Ergebnis, Interpretation und Fazit

Dieses Notebook erzeugt vier Setup-Artefakte in `data/processed/tabpfn/00_setup/`:

- `model_data_split_overview.csv`
- `target_distribution_by_split.csv`
- `feature_columns.csv`
- `legacy_artifact_overview.csv`

Diese Dateien sind keine Modelloutputs, sondern methodische Kontrolltabellen. Sie sollten vor API-Läufen überprüft werden, damit spätere TabPFN-Ergebnisse sauber auf die dokumentierte Datenbasis zurückgeführt werden können.